# 01. Caracterización de datos

Consulta el perfilamiento de Bronze generado por `src/caracterizar_bookeau.py` y `src/documentar_caracterizacion.py` (pasos `caracterizar` y `documentar` de `main.py`). Resultados en `docs/artifacts/caracterizacion/`.

Con los datos reales: 114 libros de reservas y encuestas, 86.562 reservas con ID único y 60.390 encuestas vinculadas (146.952 filas en total, que no equivalen a reservas).

**Antes de ejecutar:** desde la raíz del proyecto, correr `python3 src/main.py --dummy` (datos ficticios versionados) o `python3 src/main.py` (datos reales, que no se versionan). El cuaderno solo lee los resultados del pipeline en `docs/artifacts/`, `data/plata/` y `data/oro/`. Con datos ficticios los conteos son pequeños y no coinciden con las cifras del informe, que usan los datos reales.


In [1]:
from pathlib import Path
import csv
from html import escape
from IPython.display import display, HTML

PROYECTO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "main.py").is_file()),
    None,
)
if PROYECTO is None:
    raise RuntimeError("Abre el cuaderno desde la carpeta del proyecto o desde notebooks/.")
RESULTADOS = PROYECTO / "docs" / "artifacts" / "caracterizacion"
if not RESULTADOS.is_dir():
    raise RuntimeError("Faltan los resultados del pipeline. Ejecuta primero desde la raíz: "
                       "python3 src/main.py --dummy  (o sin --dummy si tienes los datos reales)")

def cargar(nombre):
    with (RESULTADOS / nombre).open(newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))

def tabla(filas, columnas=None, limite=None):
    if not filas:
        print("Sin registros para esta selección.")
        return
    columnas = columnas or list(filas[0])
    mostrar = filas if limite is None else filas[:limite]
    cabecera = "".join(f"<th>{escape(c)}</th>" for c in columnas)
    cuerpo = "".join(
        "<tr>" + "".join(f"<td>{escape(str(fila[c]))}</td>" for c in columnas) + "</tr>"
        for fila in mostrar
    )
    display(HTML(f"<table><thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table>"))
    print(f"Mostrando {len(mostrar)} de {len(filas)} registros.")


## 1. Volumen, grano y cobertura

Una fila de Reservas corresponde a un evento de reserva. Una fila de encuesta corresponde a una respuesta asociada al evento, según el ID observado. No se deben sumar las encuestas al conteo de reservas.

| Fuente | Filas | Variables |
|---|---:|---:|
| Reservas | 86.562 | 17 |
| Encuesta Reserva Express | 13.815 | 22 |
| Encuesta Reserva Normal IP | 23.437 | 22 |
| Encuesta Express | 6.759 | 33 |
| Encuesta Normal | 16.379 | 35 |


In [2]:
tabla(cargar("resumen_grupos.csv"))
archivos = cargar("archivos.csv")
tabla([f for f in archivos if int(f["filas"]) == 0])


grupo,archivos,filas,variables,variantes_encabezado,id_distintos,id_faltantes,filas_excedentes_id_repetido,filas_excedentes_id_periodo_repetido,filas_excedentes_duplicado_exacto,inicio_min,inicio_max,duracion_programada_min_minutos,duracion_programada_mediana_minutos,duracion_programada_max_minutos
Encuesta Express,29,6759,33,1,6759,0,0,0,0,2017-02-02 10:30:00.000000,2026-05-22 15:00:00.000000,20.0,30.0,50.0
Encuesta Normal,29,16379,35,1,16379,0,0,0,0,2016-08-19 09:00:00.000000,2026-05-22 17:00:00.000000,50.0,55.0,120.0
Encuesta Reserva Express,14,13815,22,1,13815,0,0,0,0,2019-10-21 11:00:00.000000,2026-05-22 16:30:00.000000,30.0,30.0,30.0
Encuesta Reserva Normal IP,13,23437,22,1,23437,0,0,0,0,2019-10-21 09:00:00.000000,2026-05-22 17:00:00.000000,50.0,60.0,60.0
Reservas,29,86562,17,1,86562,0,0,0,0,2016-08-19 09:00:00.000000,2026-05-22 17:00:00.000000,20.0,50.0,120.0


Mostrando 5 de 5 registros.


grupo,archivo,hoja,periodo,filas,variables
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201620.xlsx,Respuestas,201620,0,33
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 202219.xlsx,Respuestas,202219,0,33
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 202319.xlsx,Respuestas,202319,0,33
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 202419.xlsx,Respuestas,202419,0,33
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 202519.xlsx,Respuestas,202519,0,33
Encuesta Normal,data/bronce/Bookeau/Encuesta Normal/Encuesta Estudiante Normal 202219.xlsx,Respuestas,202219,0,35


Mostrando 6 de 6 registros.


## 2. Variables y significado inicial

Las descripciones son interpretaciones de los encabezados y del README. Confirmar reglas operativas, escalas y catálogos con el responsable de la fuente. Consultar `diccionario_variables.md` para las preguntas completas.


In [3]:
variables = cargar("variables.csv")
diccionario = cargar("diccionario_variables.csv")
tabla([v for v in diccionario if v["grupo"] == "Reservas"],
      ["variable", "descripcion_inicial", "tipos_observados", "porcentaje_faltantes", "valores_distintos"])
# Cambiar el grupo para explorar las preguntas y tipos de cada encuesta.
GRUPO = "Encuesta Express"
tabla([v for v in diccionario if v["grupo"] == GRUPO],
      ["variable", "rol", "tipos_observados", "porcentaje_faltantes", "primer_periodo_con_valores"])


variable,descripcion_inicial,tipos_observados,porcentaje_faltantes,valores_distintos
ID,Identificador del evento de reserva. Único en cada grupo; todas las encuestas lo comparten con Reservas.,"{""int"": 86562}",0.0,86562
Fecha inicio,"Inicio programado del evento, interpretado a partir del encabezado y del formato Excel.","{""datetime"": 86562}",0.0,16834
Fecha fin,Fin programado del evento. No equivale a una salida real registrada.,"{""datetime"": 86562}",0.0,16970
Fecha de llegada,Marca temporal de llegada según el encabezado; confirmar procedimiento de captura y excepciones.,"{""datetime"": 43140}",50.1629,42867
Fecha de devolución,Campo temporal sin valores en todos los archivos; su significado operativo debe confirmarse.,{},100.0,0
Estado de la reserva,"Estado del evento de reserva, con 12 etiquetas en la fuente principal. No confundir con Estado de la encuesta.","{""str"": 86562}",0.0,12
Prioritaria,Indicador sí/no de prioridad de la reserva; confirmar regla de asignación.,"{""str"": 86562}",0.0,2
Categoría,"Categoría de recurso o prestación, como Tutor Presencial; incluye identificadores históricos Deprecated.","{""str"": 86562}",0.0,11
Tipo de horario,"Modalidad o tipo de agenda: Normal, Express, Normal pico y otras modalidades históricas.","{""str"": 86562}",0.0,10
Servicio,"Servicio o curso asociado: IP, EDA y etiquetas históricas, entre otros.","{""str"": 86562}",0.0,11


Mostrando 17 de 17 registros.


variable,rol,tipos_observados,porcentaje_faltantes,primer_periodo_con_valores
ID,identificador,"{""int"": 6759}",0.0,201710
Fecha inicio,fecha/hora,"{""datetime"": 6759}",0.0,201710
Fecha fin,fecha/hora,"{""datetime"": 6759}",0.0,201710
Fecha de llegada,fecha/hora,"{""datetime"": 6757}",0.0296,201710
Fecha de devolución,fecha/hora,{},100.0,
Estado de la reserva,categoría o respuesta,"{""str"": 6759}",0.0,201710
Prioritaria,categoría o respuesta,"{""str"": 6759}",0.0,201710
Categoría,categoría o respuesta,"{""str"": 6759}",0.0,201710
Tipo de horario,categoría o respuesta,"{""str"": 6759}",0.0,201710
Servicio,categoría o respuesta,"{""str"": 6759}",0.0,201710


Mostrando 33 de 33 registros.


## 3. Completitud y coherencia temporal

`Fecha de devolución` está vacía en todas las fuentes: no hay base para calcular duración efectiva. `Fecha de llegada` falta en el 50,16% de Reservas; se debe interpretar junto con el estado. Algunas preguntas no tienen respuestas en los primeros períodos, aunque sus encabezados estén presentes.


In [4]:
tabla([v for v in variables if v["grupo"] == "Reservas"],
      ["variable", "filas", "no_vacios", "faltantes", "porcentaje_faltantes"])
tabla(cargar("estado_llegada.csv"))
tabla(cargar("reglas_fechas.csv"))
tabla(cargar("reglas_respuestas.csv"))


variable,filas,no_vacios,faltantes,porcentaje_faltantes
ID,86562,86562,0,0.0
Fecha inicio,86562,86562,0,0.0
Fecha fin,86562,86562,0,0.0
Fecha de llegada,86562,43140,43422,50.1629
Fecha de devolución,86562,0,86562,100.0
Estado de la reserva,86562,86562,0,0.0
Prioritaria,86562,86562,0,0.0
Categoría,86562,86562,0,0.0
Tipo de horario,86562,86562,0,0.0
Servicio,86562,86562,0,0.0


Mostrando 17 de 17 registros.


estado_reserva,filas,con_llegada,sin_llegada,porcentaje_con_llegada
Cancelada,1764,0,1764,0.0
Cancelada a tiempo,18147,4,18143,0.022
Cancelada con excusa,784,16,768,2.0408
Cancelada por sanción,2523,8,2515,0.3171
Cancelada y sancionada,286,0,286,0.0
Cola cancelada por reservación,4603,4,4599,0.0869
En cola,10215,0,10215,0.0
En ejecución,307,307,0,100.0
Finalizada,38541,38533,8,99.9792
No asistió,5086,9,5077,0.177


Mostrando 12 de 12 registros.


grupo,regla,evaluables,incumplimientos,porcentaje_evaluables
Encuesta Express,fin_anterior_a_inicio,6759,0,0.0
Encuesta Express,devolucion_anterior_a_llegada,0,0,
Encuesta Express,inicio_fuera_anio_periodo,6759,0,0.0
Encuesta Normal,fin_anterior_a_inicio,16379,0,0.0
Encuesta Normal,devolucion_anterior_a_llegada,0,0,
Encuesta Normal,inicio_fuera_anio_periodo,16379,0,0.0
Encuesta Reserva Express,fin_anterior_a_inicio,13815,0,0.0
Encuesta Reserva Express,devolucion_anterior_a_llegada,0,0,
Encuesta Reserva Express,inicio_fuera_anio_periodo,13815,0,0.0
Encuesta Reserva Normal IP,fin_anterior_a_inicio,23437,0,0.0


Mostrando 15 de 15 registros.


grupo,variable,regla,no_vacios_evaluados,fuera_dominio,faltantes
Encuesta Express,Califique la ayuda que le dio su tutor,valor entero entre 1 y 5 (dominio candidato observado),4983,0,1776
Encuesta Normal,Califique la ayuda que le dio su tutor,valor entero entre 1 y 5 (dominio candidato observado),11326,0,5053


Mostrando 2 de 2 registros.


## 4. Identificadores y vínculos

Todos los ID de encuesta existen en Reservas y tienen el mismo período. No hay ID repetidos dentro de cada grupo. Comparar los atributos repetidos antes de decidir la fuente de referencia.


In [5]:
tabla(cargar("vinculos_reservas_encuestas.csv"))
tabla([r for r in cargar("consistencia_entre_fuentes.csv") if int(r["diferencias"]) > 0])


grupo_encuesta,filas_con_id,id_distintos,id_en_reservas,filas_con_id_en_reservas,filas_sin_id_en_reservas,filas_con_id_y_periodo_en_reservas,porcentaje_filas_con_id_en_reservas
Encuesta Express,6759,6759,6759,6759,0,6759,100.0
Encuesta Normal,16379,16379,16379,16379,0,16379,100.0
Encuesta Reserva Express,13815,13815,13815,13815,0,13815,100.0
Encuesta Reserva Normal IP,23437,23437,23437,23437,0,23437,100.0


Mostrando 4 de 4 registros.


grupo_encuesta,variable,pares_no_vacios_comparados,diferencias,porcentaje_diferencias
Encuesta Express,Estado de la reserva,6759,5,0.074
Encuesta Express,Usuario,6759,1,0.0148
Encuesta Express,Código usuario,6759,5,0.074
Encuesta Express,Programa,6740,12,0.178
Encuesta Express,Prestador del servicio,6757,67,0.9916
Encuesta Normal,Estado de la reserva,16379,17,0.1038
Encuesta Normal,Usuario,16379,3,0.0183
Encuesta Normal,Código usuario,16379,10,0.0611
Encuesta Normal,Programa,16338,35,0.2142
Encuesta Normal,Prestador del servicio,16377,257,1.5693


Mostrando 20 de 20 registros.


## 5. Categorías y dominios

`Estado` es la validez de una encuesta; `Estado de la reserva` describe el evento. No confundirlos al definir indicadores. Las variantes de tildes y mayúsculas son candidatos de homologación; los catálogos Deprecated requieren explicación histórica.


In [6]:
categorias = cargar("categorias.csv")
tabla([r for r in categorias if r["grupo"] == "Reservas" and r["variable"] in
       {"Estado de la reserva", "Tipo de horario", "Servicio"}])
tabla([r for r in categorias if r["variable"] == "Estado"])
tabla([r for r in cargar("variantes_categoricas.csv") if r["grupo"] == "Reservas"])


grupo,variable,valor,frecuencia,porcentaje_filas
Reservas,Estado de la reserva,Finalizada,38541,44.5242
Reservas,Estado de la reserva,Cancelada a tiempo,18147,20.9642
Reservas,Estado de la reserva,En cola,10215,11.8008
Reservas,Estado de la reserva,No asistió,5086,5.8756
Reservas,Estado de la reserva,Cola cancelada por reservación,4603,5.3176
Reservas,Estado de la reserva,Realizada,4105,4.7423
Reservas,Estado de la reserva,Cancelada por sanción,2523,2.9147
Reservas,Estado de la reserva,Cancelada,1764,2.0378
Reservas,Estado de la reserva,Cancelada con excusa,784,0.9057
Reservas,Estado de la reserva,En ejecución,307,0.3547


Mostrando 33 de 33 registros.


grupo,variable,valor,frecuencia,porcentaje_filas
Encuesta Express,Estado,Válida,6706,99.2159
Encuesta Express,Estado,Inválida,53,0.7841
Encuesta Normal,Estado,Válida,16268,99.3223
Encuesta Normal,Estado,Inválida,111,0.6777
Encuesta Reserva Express,Estado,Válida,13815,100.0
Encuesta Reserva Normal IP,Estado,Válida,23435,99.9915
Encuesta Reserva Normal IP,Estado,Inválida,2,0.0085


Mostrando 7 de 7 registros.


grupo,variable,valor_normalizado_candidato,variantes_originales_json,filas
Reservas,Programa,ingenieria industrial,"[[""Ingenieria Industrial"", 1], [""INGENIERIA INDUSTRIAL"", 19198], [""INGENIERÍA INDUSTRIAL"", 252], [""Ingeniería Industrial"", 91]]",19542
Reservas,Programa,ingenieria mecanica,"[[""INGENIERIA MECANICA"", 4869], [""Ingeniería Mecánica"", 32], [""Ingenieria Mecanica"", 28], [""INGENIERÍA MECANICA"", 34]]",4963
Reservas,Programa,ingenieria electrica,"[[""INGENIERIA ELECTRICA"", 1526], [""INGENIERÍA ELECTRICA"", 45], [""Ingenieria Electrica"", 1]]",1572
Reservas,Programa,ingenieria civil,"[[""INGENIERIA CIVIL"", 1681], [""INGENIERÍA CIVIL"", 4046], [""Ingeniería Civil"", 62], [""Ingenieria Civil"", 3]]",5792
Reservas,Programa,ingenieria electronica,"[[""INGENIERIA ELECTRONICA"", 3342], [""Ingeniería Electrónica"", 123], [""Ingenieria Electronica"", 25], [""INGENIERÍA ELECTRONICA"", 114]]",3604
Reservas,Programa,ingenieria quimica,"[[""INGENIERIA QUIMICA"", 6041], [""INGENIERÍA QUIMICA"", 10], [""Ingenieria Quimica"", 4], [""Ingeniería Química"", 7]]",6062
Reservas,Programa,geociencias,"[[""GEOCIENCIAS"", 1993], [""Geociencias"", 16]]",2009
Reservas,Programa,estudios dirigidos,"[[""ESTUDIOS DIRIGIDOS"", 1519], [""Estudios Dirigidos"", 19]]",1538
Reservas,Programa,ingenieria ambiental,"[[""INGENIERIA AMBIENTAL"", 2609], [""Ingeniería Ambiental"", 8]]",2617
Reservas,Programa,fisica,"[[""FISICA"", 3561], [""Física"", 86], [""Fisica"", 22]]",3669


Mostrando 20 de 20 registros.


## 6. Texto libre

Las encuestas contienen comentarios sobre tutor y tutoría, observaciones del curso, sugerencias y descripciones del uso de IA. Se contabilizan respuestas no vacías y longitudes; no se asume que toda respuesta sea sustantiva.


In [7]:
tabla(cargar("texto_libre.csv"))


grupo,variable,respuestas_no_vacias,porcentaje_cobertura,textos_distintos,longitud_min,longitud_mediana,longitud_max
Encuesta Express,Indique cuál fue la herramienta usada durante la tutoría (si sabe cuál es) y el uso que tuvo la misma en la tutoría.,4,0.0592,3,6,16.0,238
Encuesta Express,Comente aspectos positivos y negativos del tutor y la tutoría. Recuerde que las respuestas van a ser anónimas.,5453,80.6776,3878,1,26,550
Encuesta Express,Escriba en este recuadro cualquier observación o comentario relacionado al curso y sus contenidos.,564,8.3444,265,1,7.0,339
Encuesta Express,"Sugerencias, reclamos y observaciones",1730,25.5955,655,1,7.0,450
Encuesta Normal,Indique cuál fue la herramienta usada durante la tutoría (si sabe cuál es) y el uso que tuvo la misma en la tutoría.,22,0.1343,20,1,14.5,90
Encuesta Normal,Comente aspectos positivos y negativos del tutor y la tutoría. Recuerde que las respuestas van a ser anónimas.,12534,76.5248,8526,1,29.0,1912
Encuesta Normal,Escriba en este recuadro cualquier observación o comentario relacionado al curso y sus contenidos.,1329,8.114,580,1,7,878
Encuesta Normal,"Sugerencias, reclamos y observaciones",4053,24.7451,1411,1,7,2281


Mostrando 8 de 8 registros.


## 7. Cobertura por período

La primera respuesta observada no demuestra cuándo se creó la pregunta. Antes de comparar satisfacción históricamente, revisar formularios y denominadores de respuesta.


In [8]:
cobertura = cargar("cobertura_variables_periodo.csv")
PREGUNTA = "Califique la ayuda que le dio su tutor"
tabla([r for r in cobertura if r["grupo"] == GRUPO and r["variable"] == PREGUNTA])


grupo,periodo,variable,filas,no_vacios,porcentaje_no_vacios
Encuesta Express,201620,Califique la ayuda que le dio su tutor,0,0,
Encuesta Express,201710,Califique la ayuda que le dio su tutor,781,0,0.0
Encuesta Express,201719,Califique la ayuda que le dio su tutor,10,0,0.0
Encuesta Express,201720,Califique la ayuda que le dio su tutor,487,0,0.0
Encuesta Express,201810,Califique la ayuda que le dio su tutor,498,0,0.0
Encuesta Express,201819,Califique la ayuda que le dio su tutor,17,17,100.0
Encuesta Express,201820,Califique la ayuda que le dio su tutor,594,594,100.0
Encuesta Express,201910,Califique la ayuda que le dio su tutor,614,614,100.0
Encuesta Express,201919,Califique la ayuda que le dio su tutor,21,21,100.0
Encuesta Express,201920,Califique la ayuda que le dio su tutor,381,381,100.0


Mostrando 29 de 29 registros.


## 8. Regenerar la caracterización

Las celdas anteriores leen los resultados existentes. La siguiente celda vuelve a leer todos los Excel y genera las tablas y documentos; activarla cuando se incorporen fuentes nuevas. Usa la biblioteca estándar de Python y conserva los originales.


In [9]:
# import runpy
# runpy.run_path(str(PROYECTO / "src" / "caracterizar_bookeau.py"), run_name="__main__")
# runpy.run_path(str(PROYECTO / "src" / "documentar_caracterizacion.py"), run_name="__main__")


## Decisiones pendientes

Confirmar estados de reserva, reglas de asistencia, códigos de período, categorías históricas, identidad de participantes, escalas y formularios históricos, y criterios de invalidez de encuestas.

La fuente permite explorar demanda y satisfacción. No incluye notas académicas, capacidad disponible explícita ni costos para demostrar impacto causal de tutorías, ocupación de capacidad o rentabilidad.
